# DeepSTARR for multi-task enhancer design

[Notebook](multi-task.ipynb) · [Installation](../docs/installation.md)

Decrease predicted developmental activity and increase predicted housekeeping activity in 249-nt enhancers. Install MEME Suite with the notebook dependencies. The first model load downloads the pretrained checkpoint.

## 1. Load the model



In [1]:
# %pip install "bioseq-models @ git+https://github.com/UTR-CODE/bioseq-models.git@84a5075c3656127d99027c38705212a2de6e62c9"

from pathlib import Path
from tempfile import mkdtemp

import torch
from bioseq_models import load_model
from MotifSift import MultiTaskObjective, call_motifs, generate_designs, random_seed_sequences, select_candidates

device = "cuda" if torch.cuda.is_available() else "cpu"
model = load_model("deepstarr", device=device)
objective = MultiTaskObjective(high=["housekeeping"], low=["developmental"])
seeds = random_seed_sequences(249, 100, seed=42)
run = Path(mkdtemp(prefix="motifsift-example-"))




## 2. Generate candidates

Run the eleven optimizers and keep 100 candidates per optimizer. The saved tables show the previous full run. New runs use the current MotifSift defaults.



In [2]:
designs = generate_designs(
    model, length=249, initial=seeds, objective=objective,
    output_dir=run / "designs",
)
designs.groupby("algorithm")["score"].agg(["size", "median", "max"])


,size,median,max
algorithm,,,
adalead,100,1.710509,1.748892
basesearch,100,1.586460,1.699707
beam_search,100,1.562266,1.821398
cross_entropy_method,100,1.541474,1.626658
den,100,1.715076,1.778538
fast_seqprop,100,1.772594,1.878561
genetic_algorithm,100,1.785317,1.843402
ism_greedy,100,1.532103,1.738242
ledidi,100,1.478913,1.653784




## 3. Discover motifs

Extract positive seqlets from in silico mutagenesis scores and discover motifs with XSTREME.



In [3]:
motifs = call_motifs(
    designs, model=model,
    output_dir=run / "motifs",
)
motifs.support.query("retained")[["motif_id", "seqlets", "parents", "sites_on_unique_DNA"]]


,motif_id,seqlets,parents,sites_on_unique_DNA
0,1-ACTATCGATWG,838,688,2048.0
1,2-CAGTGTGGCC,134,132,339.0
2,3-AGVGATGSM,126,115,41.0
14,29-CGATATATCGT,55,55,56.0




## 4. Select a panel



In [4]:
gate = float(designs.drop_duplicates("sequence").score.quantile(0.2))
selected = select_candidates(designs, motifs=motifs, top_n=10, min_score=gate, output_dir=run / "selected")
selected[["selection_rank", "sequence", "score", "present_motifs"]]


,selection_rank,sequence,score,present_motifs
0,1,GTCCTGCCTCCACCGCGGGCTCCAGGTCGAGGGTCTCGACCGTTAC...,1.925456,1-ACTATCGATWG
28,2,CGTCGGCCTCCGGTCCGGGGGGGGGGGCCGGTTCTCCCGACGTTGA...,1.827626,2-CAGTGTGGCC
646,3,GACGCGTGCCGATATATCGCTATGTAGCCCTTTAGAGACCGTTCGC...,1.572216,3-AGVGATGSM;29-CGATATATCGT
516,4,CTCCTAGTTGAGCTGAGGGATACGGAAGGACCGTCTTGAGTGTTCA...,1.677060,1-ACTATCGATWG;2-CAGTGTGGCC;3-AGVGATGSM
700,5,AATTCTGTTGCGCCGGGCCAGGTGCAACTCCTACACGACATGAAAC...,1.558585,29-CGATATATCGT
848,6,GTCCTAACTGAGCTGCGGGCGACCGTTCGACGGTCTTGACTGTTAC...,1.509491,3-AGVGATGSM
837,7,GTTGCGGGCCGATATATCGATATGTAACCTTCTAGAGCCCCTTCCA...,1.516301,1-ACTATCGATWG;2-CAGTGTGGCC;29-CGATATATCGT
299,8,AGTAGCTAACGCGCGTTGCACCCTTGGAAAAATATCGAGCCAATCT...,1.728552,2-CAGTGTGGCC;3-AGVGATGSM
236,9,CTAACGTTTTAAAAATTTACAGGGATGGAATTCGTAGCTGTAAAAT...,1.747340,1-ACTATCGATWG;3-AGVGATGSM
614,10,TACGAAAAATCAGTTTTGGATGATAGAAGATATATCTCAGTATATA...,1.589044,1-ACTATCGATWG;29-CGATATATCGT
